# DEVELOPMENT: DTU ORBIT Meta Data

## Cleaning the Metadata from DTU ORBIT

### Loading the data

In [1]:
import pandas as pd
import IPython

df_orbit = pd.read_csv('../Data/Orbit_meta/dtu_orbit_persons_raw.csv')
pd.set_option("display.max_colwidth", None)  # Show full text in cells

# making everything in the dataset lowercase
df_orbit = df_orbit.map(lambda x: x.lower() if isinstance(x, str) else x)

# Professor, National Institute of Aquatic Resources
display(df_orbit.head(2))
print("Number of rows in raw dataset: ", len(df_orbit))

,url,name,affiliations,email,orcid,website,address,profile_text,keywords,sdgs
0,https://orbit.dtu.dk/en/persons/a-s-m-lutfor-rahman-rabbi/,a s m lutfor rahman rabbi,"teaching assistant, department of technology, management and economics",asmlur@dtu.dk,NaN,http://www.man.dtu.dk,denmark,NaN,NaN,NaN
1,https://orbit.dtu.dk/en/persons/aage-thaarup/,aage thaarup,"fisheries technician, national institute of aquatic resources; section for monitoring and data",att@aqua.dtu.dk,NaN,http://www.aqua.dtu.dk,"willemoesvej 2 , hovedbygning, 068 9850 hirtshals denmark",NaN,NaN,NaN


Number of rows in raw dataset:  5734


### Splitting affiliations into: ``professional_title`` & ``dtu_department``

In [2]:
# Split affiliations into:
# 1) professional_title (before first comma)
# 2) dtu_department (after first comma)
parts = df_orbit["affiliations"].astype("string").str.split(",", n=1, expand=True)

df_orbit["professional_title"] = parts[0].str.strip()
df_orbit["dtu_department"] = parts[1].str.strip()

# If no comma existed, keep department as missing value (already NaN/<NA>)
df_orbit["dtu_department"] = df_orbit["dtu_department"].fillna(pd.NA)

# Quick check
display(df_orbit[["affiliations", "professional_title", "dtu_department"]].head(10))
print("Rows with missing title:", df_orbit["professional_title"].isna().sum())
print("Rows with missing department:", df_orbit["dtu_department"].isna().sum())

,affiliations,professional_title,dtu_department
0,"teaching assistant, department of technology, management and economics",teaching assistant,"department of technology, management and economics"
1,"fisheries technician, national institute of aquatic resources; section for monitoring and data",fisheries technician,national institute of aquatic resources; section for monitoring and data
2,"postdoc, department of civil and mechanical engineering; manufacturing engineering",postdoc,department of civil and mechanical engineering; manufacturing engineering
3,"postdoc, national food institute; research group for food allergy",postdoc,national food institute; research group for food allergy
4,"metabolomics core manager, department of biotechnology and biomedicine; section for microbial and chemical ecology; dtu metabolomics core; natural product discovery; center for microbial secondary metabolites",metabolomics core manager,department of biotechnology and biomedicine; section for microbial and chemical ecology; dtu metabolomics core; natural product discovery; center for microbial secondary metabolites
5,"student assistant, department of electrical and photonics engineering",student assistant,department of electrical and photonics engineering
6,"professor, department of applied mathematics and computer science; visual computing",professor,department of applied mathematics and computer science; visual computing
7,"postdoc, department of applied mathematics and computer science; cognitive systems",postdoc,department of applied mathematics and computer science; cognitive systems
8,"phd student, department of energy conversion and storage; structural analysis and modelling",phd student,department of energy conversion and storage; structural analysis and modelling
9,"phd student, department of technology, management and economics",phd student,"department of technology, management and economics"


Rows with missing title: 4
Rows with missing department: 11


### Keeping only titles allowed to supervise

In [3]:
# Filter: who can be main supervisor (hovedvejleder) per DTU rules
# Job categories: adjunkt, lektor, docent, ingeniørdocent, professor, forsker, seniorforsker, seniorrådgiver
SUPERVISOR_TITLES = {
    'adjunkt', 'lektor', 'docent', 'ingeniørdocent', 'professor', 'forsker', 'seniorforsker', 'seniorrådgiver',
    'assistant professor', 'associate professor', 'professor', 'researcher', 'senior researcher', 'senior adviser', 'senior advisor'
}

# Normalize title column before filtering (safe)
df_orbit["professional_title"] = df_orbit["professional_title"].astype("string").str.strip()

before = len(df_orbit)
df_orbit_cut = df_orbit[df_orbit["professional_title"].isin(SUPERVISOR_TITLES)].copy()
after = len(df_orbit_cut)

print(f"Rows removed: {before - after}")
print(f"Rows remaining: {after}")


Rows removed: 4846
Rows remaining: 888


In [4]:
display(df_orbit_cut[["affiliations", "professional_title", "dtu_department"]])

,affiliations,professional_title,dtu_department
6,"professor, department of applied mathematics and computer science; visual computing",professor,department of applied mathematics and computer science; visual computing
10,"researcher, department of physics; 2d materials engineering and physics",researcher,department of physics; 2d materials engineering and physics
15,"researcher, national food institute; research group for food allergy",researcher,national food institute; research group for food allergy
16,"associate professor, department of health technology; hearing systems section; computational auditory modeling",associate professor,department of health technology; hearing systems section; computational auditory modeling
19,"associate professor, national centre for nano fabrication and characterization; nanofabrication; polymer microsystems",associate professor,national centre for nano fabrication and characterization; nanofabrication; polymer microsystems
...,...,...,...
5686,"senior researcher, department of civil and mechanical engineering; manufacturing engineering",senior researcher,department of civil and mechanical engineering; manufacturing engineering
5687,"associate professor, department of environmental and resource engineering; geotechnics & geology",associate professor,department of environmental and resource engineering; geotechnics & geology
5691,"senior researcher, department of electrical and photonics engineering; high-speed optical communications; nanophotonic devices; centre of excellence for silicon photonics for optical communications",senior researcher,department of electrical and photonics engineering; high-speed optical communications; nanophotonic devices; centre of excellence for silicon photonics for optical communications
5697,"associate professor, department of environmental and resource engineering; waste, climate & monitoring",associate professor,"department of environmental and resource engineering; waste, climate & monitoring"


In [5]:
display(df_orbit_cut["professional_title"].unique())

<StringArray>
[          'professor',          'researcher', 'associate professor',
   'senior researcher', 'assistant professor',      'senior adviser',
      'senior advisor',              'lektor']
Length: 8, dtype: string

### Splitting dtu_department into ``department`` and ``section``

In [6]:
# Split dtu_department into:
# 1) department (before first semicomma)
# 2) section (after first semicomma)
parts = df_orbit_cut["dtu_department"].astype("string").str.split(";", n=1, expand=True)

df_orbit_cut["department"] = parts[0].str.strip()
df_orbit_cut["section"] = parts[1].str.strip()

# If no semicomma existed, keep department as missing value (already NaN/<NA>)
df_orbit_cut["section"] = df_orbit_cut["section"].fillna(pd.NA)

# Quick check
display(df_orbit_cut[["dtu_department", "department", "section"]].head(10))
print("Rows with missing department:", df_orbit_cut["department"].isna().sum())
print("Rows with missing section:", df_orbit_cut["section"].isna().sum())

,dtu_department,department,section
6,department of applied mathematics and computer science; visual computing,department of applied mathematics and computer science,visual computing
10,department of physics; 2d materials engineering and physics,department of physics,2d materials engineering and physics
15,national food institute; research group for food allergy,national food institute,research group for food allergy
16,department of health technology; hearing systems section; computational auditory modeling,department of health technology,hearing systems section; computational auditory modeling
19,national centre for nano fabrication and characterization; nanofabrication; polymer microsystems,national centre for nano fabrication and characterization,nanofabrication; polymer microsystems
23,department of environmental and resource engineering; water technology & processes,department of environmental and resource engineering,water technology & processes
29,department of civil and mechanical engineering; energy and services,department of civil and mechanical engineering,energy and services
32,national food institute; research group for bioactives – analysis and application,national food institute,research group for bioactives – analysis and application
38,department of physics; quantum physics and information technology,department of physics,quantum physics and information technology
45,department of civil and mechanical engineering; thermal energy,department of civil and mechanical engineering,thermal energy


Rows with missing department: 0
Rows with missing section: 35


In [7]:
import json
import re
import pandas as pd

# Load DTU departments JSON
with open('../Data/Departments_DTU_all.json', 'r', encoding='utf-8') as f:
    dep = json.load(f)
print(f"Loaded {len(dep)} departments from JSON file.")

def _flatten_text(value):
    """Return a flat list of strings from nested str/list/dict structures."""
    out = []
    if value is None:
        return out
    if isinstance(value, str):
        out.extend([v.strip() for v in value.split("|") if v.strip()])
    elif isinstance(value, list):
        for v in value:
            out.extend(_flatten_text(v))
    elif isinstance(value, dict):
        for v in value.values():
            out.extend(_flatten_text(v))
    return out

def _norm(s):
    s = str(s).strip().lower()
    s = re.sub(r"^dtu\s+", "", s)   # remove leading "dtu "
    s = re.sub(r"\s+", " ", s)      # normalize spaces
    return s

def _department_value(dep_item):
    d = dep_item.get("department")
    if isinstance(d, dict):
        return d.get("en") or d.get("da") or next((str(v) for v in d.values() if v), None)
    return d

# Build alias -> canonical department lookup from title + sections
alias_to_department = {}

for item in dep:
    department_val = _department_value(item)
    aliases = []
    aliases.extend(_flatten_text(item.get("title")))
    aliases.extend(_flatten_text(item.get("sections")))

    for a in aliases:
        alias_to_department[_norm(a)] = department_val

# Map df_orbit_cut["department"] to canonical department
def map_department_to_json_match(department):
    if pd.isna(department):
        return pd.NA

    d_norm = _norm(department)

    # exact match first
    if d_norm in alias_to_department:
        return alias_to_department[d_norm]

    # fallback: contains match either direction
    for alias, dep_val in alias_to_department.items():
        if alias in d_norm or d_norm in alias:
            return dep_val

    return pd.NA

df_orbit_cut["dep_match"] = df_orbit_cut["department"].apply(map_department_to_json_match)

print("Matched rows:", df_orbit_cut["dep_match"].notna().sum(), "/", len(df_orbit_cut))
display(df_orbit_cut[["department", "dep_match"]].head(20))

# Optional: inspect unmatched departments
unmatched_departments = (
    df_orbit_cut.loc[df_orbit_cut["dep_match"].isna(), "department"]
    .fillna("Missing")
    .astype(str)
    .str.strip()
    .replace("", "Missing")
    .value_counts()
    .rename_axis("department")
    .reset_index(name="count")
)

print(f"Unmatched rows: {unmatched_departments['count'].sum()} / {len(df_orbit_cut)}")
display(unmatched_departments.head(30))

Loaded 20 departments from JSON file.
Matched rows: 876 / 888


,department,dep_match
6,department of applied mathematics and computer science,DTU Compute
10,department of physics,DTU Physics
15,national food institute,DTU Food
16,department of health technology,DTU Health Tech
19,national centre for nano fabrication and characterization,DTU Nanolab
23,department of environmental and resource engineering,DTU Sustain
29,department of civil and mechanical engineering,DTU Construct
32,national food institute,DTU Food
38,department of physics,DTU Physics
45,department of civil and mechanical engineering,DTU Construct


Unmatched rows: 12 / 888


,department,count
0,dtu admission course,7
1,it service,2
2,office for study programmes and student affairs,1
3,"group leader, novo nordisk foundation center for biosustainability",1
4,center for teaching and learning in engineering education,1


### Dropping all rows with *N/A* in ``dep_match``

In [8]:
df_orbit_clean = df_orbit_cut[~df_orbit_cut["dep_match"].isna()].copy()
print("The number of abailable supervisors are;", len(df_orbit_clean))

The number of abailable supervisors are; 876


### Exporting the cleaned df

In [12]:
export_true = input("Do you want to export the cleaned DataFrame? (yes/no): ").strip().lower()

if export_true == "yes":
    df_orbit_clean.to_csv('../Data/Orbit_meta/Cleaned/dtu_orbit_persons_cleaned.csv', index=False)
    print("Cleaned DataFrame exported to '../Data/Orbit_meta/dtu_orbit_persons_cleaned.csv'")
else:
    print("Export skipped.")

Cleaned DataFrame exported to '../Data/Orbit_meta/dtu_orbit_persons_cleaned.csv'
